# 14 — Monitoring queries for the deployed PolicyPal endpoint

Production-monitoring companion notebook. Lands the
SQL behind the chapter's two figures plus the supporting queries the
prose teaches.

**What this notebook runs:**

* §1 — Cost attribution per endpoint via `system.serving.endpoint_usage`
  joined to `system.serving.served_entities` (cost-attribution dashboard source query).
* §2 — Per-team cost via `system.ai_gateway.usage` aggregated on the
  `endpoint_tags['team']` map subscript.
* §3 — Rate-limit hit rate per principal via `status_code = 429` on the
  chain endpoint's inference table.
* §4 — PII-guardrail block counts via `status_code = 400` on
  `system.ai_gateway.usage` (with the Custom-Model-Serving caveat).
* §5 — SLO latency percentiles (p50/p95/p99) on the chain endpoint's inference
  table.
* §6 — Agent Monitoring setup for production scorers on the live
  policypal-chain traces (agent-monitoring dashboard source). Shown as the
  documented `Scorer.register(...).start(ScorerSamplingConfig(...))`
  shape; not executed by default (the monitor schedules persistent
  scorer runs against live traffic and assessments take 15–20 min
  to appear after first activation).

## How to run it

1. Set the `catalog` widget (book default `genaicert`).
2. Run all cells. The notebook runs on Databricks default serverless
   compute; no cluster creation needed.
3. On a fresh workspace (no PolicyPal traffic yet), most queries
   return zero rows — the **query shape** is the teaching point, not
   the row count. Re-run after the PolicyPal endpoint has served real
   traffic for a day to see meaningful aggregates.

## Source verification

* `system.serving.endpoint_usage` + `system.serving.served_entities`
  columns — verified May 2026 against the
  [billable usage docs](https://docs.databricks.com/aws/en/admin/account-settings/billable-usage-delivery).
* `system.ai_gateway.usage` schema + `endpoint_tags` map subscript —
  verified May 2026 against the
  [usage-tracking-beta docs](https://docs.databricks.com/aws/en/ai-gateway/usage-tracking-beta).
* Agent Monitoring SDK shape — `Scorer.register(name=...)`,
  `Scorer.start(sampling_config=ScorerSamplingConfig(...))` —
  verified May 2026 against the
  [production-monitoring docs](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring).

In [0]:
dbutils.widgets.text("catalog", "genaicert")
catalog = dbutils.widgets.get("catalog")
print(f"Catalog: {catalog}")
spark.sql(f"USE CATALOG {catalog}")

Catalog: genaicert


DataFrame[]

## 1. Cost attribution per endpoint (cost-attribution dashboard source)

`system.serving.endpoint_usage` records per-request token counts for
every Model Serving endpoint; joining `served_entities` resolves the
opaque `served_entity_id` to a readable `endpoint_name`. Maria's
monthly cost-attribution dashboard reads this exact shape.

The cost-attribution SQL dashboard wraps this query with `date_trunc('week',
request_time)` for a weekly trend and a per-endpoint line chart.

In [0]:
%sql
SELECT se.endpoint_name,
       COUNT(*)                                       AS calls,
       SUM(eu.input_token_count + eu.output_token_count) AS tokens,
       SUM(eu.input_token_count)                      AS input_tokens,
       SUM(eu.output_token_count)                     AS output_tokens
FROM   system.serving.endpoint_usage eu
JOIN   system.serving.served_entities se
       ON eu.served_entity_id = se.served_entity_id
WHERE  eu.request_time >= current_date - INTERVAL 30 DAYS
GROUP  BY se.endpoint_name
ORDER  BY tokens DESC;

endpoint_name,calls,tokens,input_tokens,output_tokens
databricks-meta-llama-3-1-8b-instruct,12213,6904917,6325364,579553
databricks-gte-large-en,898,6337704,6337704,0
databricks-meta-llama-3-3-70b-instruct,70,588706,547685,41021
policypal-chain,264,0,0,0


## 2. Per-team cost via `endpoint_tags['team']`

When an endpoint is tagged at create time —
`endpoint_tags={'team': 'claims', 'cost_center': 'CC-403', 'env': 'prod'}`
— the map travels with every `system.ai_gateway.usage` row. Aggregate
monthly spend by team straight from SQL via the map subscript.

Returns zero rows if no PolicyPal endpoints carry a `team` tag.
Re-tag the `policypal-chain` endpoint via the Serving UI or
`w.serving_endpoints.patch(name=..., add_tags=[...])` to surface rows.

In [0]:
%sql
SELECT date_trunc('month', event_time) AS month,
       endpoint_tags['team']           AS team,
       SUM(input_tokens + output_tokens) AS tokens
FROM   system.ai_gateway.usage
WHERE  event_time >= current_date - INTERVAL 90 DAYS
  AND  endpoint_tags['team'] IS NOT NULL
GROUP  BY 1, 2
ORDER  BY 1, 2;

month,team,tokens


## 3. Rate-limit hit rate per principal

`status_code = 429` on the chain endpoint's inference table flags
rate-limit blocks. A consistent per-user spike when the UI shouldn't
justify it usually means a client-side retry-loop bug — fix the App,
not the rate limit.

Returns zero rows on a fresh workspace (no 429s yet).

In [0]:
%sql
SELECT requester,
       date_trunc('hour', request_time) AS hour,
       SUM(CASE WHEN status_code = 429 THEN 1 ELSE 0 END) AS rate_limit_hits,
       COUNT(*)                                            AS total_requests
FROM   monitoring.policypal_payload
WHERE  request_time >= current_date - INTERVAL 1 DAY
GROUP  BY requester, hour
HAVING rate_limit_hits > 0
ORDER  BY rate_limit_hits DESC;

requester,hour,rate_limit_hits,total_requests


## 4. PII-guardrail block counts (with PyFunc caveat)

AI Gateway guardrails return HTTP 400 with `error_code: BAD_REQUEST`.
`system.ai_gateway.usage` is the trend-and-aggregate surface.

**PolicyPal-specific caveat**: PolicyPal is Custom Model Serving
(PyFunc) — AI Gateway guardrails are *not* supported on PyFunc per
the Model Serving endpoint-type support matrix. The PII filter on PolicyPal lives
inside the chain (its `_strip_pii_before_response` step), so
`status_code = 400` on `policypal-chain` is *not* a guardrail-block
count for this endpoint. To see real guardrail blocks, point at an
FM-API pay-per-token or External-Models endpoint where guardrails fire
— example below uses the Llama 3.1 8B FM-API endpoint.

In [0]:
%sql
SELECT date_trunc('day', event_time) AS day,
       COUNT(*) FILTER (WHERE status_code = 400) AS blocked,
       COUNT(*)                                   AS total
FROM   system.ai_gateway.usage
WHERE  endpoint_name = 'databricks-meta-llama-3-1-8b-instruct'
  AND  event_time >= current_date - INTERVAL 30 DAYS
GROUP  BY 1
ORDER  BY 1;

day,blocked,total
2026-05-19T00:00:00.000Z,0,1
2026-05-22T00:00:00.000Z,0,7041
2026-05-25T00:00:00.000Z,0,1002
2026-05-26T00:00:00.000Z,0,3
2026-05-30T00:00:00.000Z,0,1
2026-06-05T00:00:00.000Z,0,2
2026-06-06T00:00:00.000Z,0,2


## 5. SLO percentile — p99 latency on the chain endpoint

PolicyPal's SLO is "99% of requests under 30s end-to-end". The alert
metric has to match the percentile — alert on `p99`, not on mean or
median. Set the alert threshold *below* the SLO (e.g., p99 > 25 s
when the SLO is 30 s) so the page fires *before* the SLO is breached.

Returns zero rows if the inference table has no requests yet.

In [0]:
%sql
SELECT date_trunc('hour', request_time)                         AS hour,
       COUNT(*)                                                 AS requests,
       percentile_approx(execution_duration_ms, 0.50) / 1000.0  AS p50_seconds,
       percentile_approx(execution_duration_ms, 0.95) / 1000.0  AS p95_seconds,
       percentile_approx(execution_duration_ms, 0.99) / 1000.0  AS p99_seconds
FROM   monitoring.policypal_payload
WHERE  request_time >= current_date - INTERVAL 7 DAYS
GROUP  BY 1
ORDER  BY 1 DESC;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7841716993424996>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "SELECT date_trunc('hour', request_time)                         AS hour,\n       COUNT(*)                                                 AS requests,\n       percentile_approx(execution_duration_ms, 0.50) / 1000.0  AS p50_seconds,\n       percentile_approx(execution_duration_ms, 0.95) / 1000.0  AS p95_seconds,\n       percentile_approx(execution_duration_ms, 0.99) / 1000.0  AS p99_seconds\nFROM   monitoring.policypal_payload\nWHERE  request_time >= current_date - INTERVAL 7 DAYS\nGROUP  BY 1\nORDER  BY 1 DESC;\n")

File /databricks/python/lib/python3.10/site-packages/IPython/core/interactiveshell.py:2478, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2476 with self.builtin_trap:
   2477     args = (magic_arg_s, cell)
-> 2478  

## 6. Agent Monitoring — production scorers on live traces (agent-monitoring dashboard source)

**Agent Monitoring** is the documented Databricks surface for running
offline-eval scorers and LLM judges automatically over production
MLflow traces. Each registered scorer evaluates a configurable
sample of incoming traces; the assessments land back on the traces
and roll up into per-scorer quality trends viewable on the MLflow
experiment's Traces tab.

The pattern is **`Scorer().register(name=...).start(sampling_config=...)`**
— no separate "enable monitor" call. Each `.start()` activates one
scorer in monitoring mode; subsequent traces matching `filter_string`
are sampled at `sample_rate` and scored.

**Source verification.** API (`Scorer.register(name=...)`,
`Scorer.start(sampling_config=ScorerSamplingConfig(...))`,
`ScorerSamplingConfig(sample_rate, filter_string)`) verified May 2026
against the [production-monitoring docs](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring).
Processing latency: assessments appear ~15–20 minutes after a trace
is sampled. Limit: 20 active scorers per experiment.

In [0]:
# Agent Monitoring — register + start scorers on production traces.
# Uncomment to activate against the live PolicyPal endpoint's traces.
#
# from mlflow.genai.scorers import (
#     Safety,
#     ScorerSamplingConfig,
#     scorer,
# )
#
# # 1. Built-in safety scorer — score 100% of traces, no filter.
# safety = Safety().register(name="policypal_safety").start(
#     sampling_config=ScorerSamplingConfig(sample_rate=1.0),
# )
#
# # 2. Custom citation-correctness scorer, sampled at 20% to keep cost in check.
# #    Use the production `citation_correctness` scorer from the c1301 evaluation
# #    notebook (every cited Section X.Y must appear in the retrieved chunks) and
# #    register + start it for monitoring the same way as the built-in above:
#
# citation = citation_correctness.register(
#     name="policypal_citation_correctness",
# ).start(
#     sampling_config=ScorerSamplingConfig(
#         sample_rate=0.2,
#         filter_string="attributes.status = 'OK'",
#     ),
# )
#
# print("Agent Monitoring activated for PolicyPal:")
# print("  - policypal_safety            (100% of traces)")
# print("  - policypal_citation_correctness (20% sample, OK responses only)")
# print("Assessments appear in the experiment's Traces tab after ~15-20 min.")

### Reading monitoring results without creating a monitor

Even without the auto-generated dashboard, the inference table itself
supports the drift-style queries by hand. Two examples worth
knowing — both run against `${catalog}.monitoring.policypal_payload`
directly.

In [0]:
%sql
-- 6a. Per-day request volume + average response length (free-form
-- proxy for "are answers getting longer / shorter / weirder?")
SELECT date_trunc('day', request_time)                  AS day,
       COUNT(*)                                          AS requests,
       AVG(LENGTH(CAST(request  AS STRING)))             AS avg_request_chars,
       AVG(LENGTH(CAST(response AS STRING)))             AS avg_response_chars
FROM   ${catalog}.monitoring.policypal_payload
WHERE  request_time >= current_date - INTERVAL 30 DAYS
GROUP  BY 1
ORDER  BY 1 DESC;

day,requests,avg_request_chars,avg_response_chars
2026-06-06T00:00:00.000Z,43,178.0,1173.5348837209303
2026-06-05T00:00:00.000Z,21,178.0,1169.7142857142858
2026-05-28T00:00:00.000Z,21,178.0,1235.1904761904761


In [0]:
%sql
-- 6b. Status-code distribution per day — surfaces shape of failures
-- (5xx errors → endpoint health; 429 → rate-limit pressure; 200 →
-- healthy serve). Pair with the §3 / §4 / §5 queries above for the
-- per-cause breakdown.
SELECT date_trunc('day', request_time) AS day,
       status_code,
       COUNT(*)                        AS requests
FROM   ${catalog}.monitoring.policypal_payload
WHERE  request_time >= current_date - INTERVAL 7 DAYS
GROUP  BY 1, 2
ORDER  BY 1 DESC, 3 DESC;

day,status_code,requests
2026-06-06T00:00:00.000Z,200,43
2026-06-05T00:00:00.000Z,200,21
